# Example 12.6.4 / 1: Solve a box counterpart

Companion notebook for **12.6.4 Robust Counterparts** in the AMPL lab Documentation, section "Solve a box counterpart". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a box counterpart

In this instance item `b` may be negative, so `abs` matters. Nominal coefficients are `(2, 1)`,
deviations `(0.5, 0.5)`, and the capacity is 10. After the solve, `ampl.get_value` evaluates the
worst-case left side at the returned `x`. HiGHS accepts `abs` because AMPL's MP interface
reformulates it.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                                              # decision items
param nominal {ITEMS};                                  # nominal capacity coefficient
param deviation {ITEMS} >= 0;                           # box half-width around the nominal value
param value {ITEMS};                                    # value per unit of x
param lower {ITEMS};                                    # lower bound on x; may be negative
param upper {ITEMS};                                    # upper bound on x
param capacity;                                         # right-hand side of the capacity row

var x {i in ITEMS} >= lower[i] <= upper[i];  # decision levels; sign not restricted in general

maximize TotalValue: sum {i in ITEMS} value[i] * x[i];  # nominal value of the plan

# Holds for every coefficient vector in the box.
subject to RobustCapacity:
    sum {i in ITEMS} nominal[i] * x[i]
  + sum {i in ITEMS} deviation[i] * abs(x[i]) <= capacity;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["a", "b"]
ampl.param["nominal"] = {"a": 2, "b": 1}
ampl.param["deviation"] = {"a": 0.5, "b": 0.5}
ampl.param["value"] = {"a": 3, "b": -1}
ampl.param["lower"] = {"a": 0, "b": -4}
ampl.param["upper"] = {"a": 6, "b": 4}
ampl.param["capacity"] = 10

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.var["x"].to_dict(), ampl.obj["TotalValue"].value())
print(ampl.get_value(
    "sum {i in ITEMS} (nominal[i] * x[i] + deviation[i] * abs(x[i]))"))

solved {'a': 4.8, 'b': -4} 18.4
10


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ITEMS := a b;
param:  nominal  deviation  value  lower  upper :=
  a           2        0.5      3      0      6
  b           1        0.5     -1     -4      4;
param capacity := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display x, TotalValue;
display sum {i in ITEMS} (nominal[i] * x[i] + deviation[i] * abs(x[i]));

HiGHS 1.14.0: optimal solution; objective 18.4
0 simplex iterations
0 barrier iterations
x [*] :=
a   4.8
b  -4
;

TotalValue = 18.4

sum{i in ITEMS} (nominal[i]*x[i] + deviation[i]*abs(x[i])) = 10

